# W15-D6 · 双 Demo 实战：Semantic Model 导入验证 × A101 依据链问答 + W16 Brief 前置证据

> 开发期 · Week 15 Day 6（2026-09-12 周六 · 实战日）
> Today's Question：**这个 Demo 距离生产环境还差几层，每层差的是什么？**
>
> 本 notebook 是可执行实验记录（md 是阅读材料，这里是证据）：
> §1 白名单合规复查（D5 遗留：11 条 SQL 示例 × 生产白名单 20 对象）
> §2 导入物化 + SQLite 镜像导入演练（starter-pack 同构格式 + 幂等收敛证明）
> §3 覆盖率 A/B 复测（21 题电池：starter 基线 vs +语义包）+ mallcre 种子真跑
> §4 Demo②：A101 依据链问答走 L1-L3（D1 判定梯机器复核）
> §5 生产差距五层分解 + S6 消费回执落盘

In [ ]:
import os, re, json, sqlite3, hashlib, subprocess, datetime
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

NB_DIR = "/root/learning-notebooks/第15周"
SM_DIR = "/root/learning-notebooks/semantic-model"
CONS_DIR = os.path.join(SM_DIR, "consumers", "lnkchatbi")
PACK_DIR = os.path.join(CONS_DIR, "import-pack")
os.makedirs(PACK_DIR, exist_ok=True)

# D6 前置（D5 遗留）：SoT 指纹复验 + lnkcre HEAD 确认（D5 已 pull 到 origin/main）
ONT_PATH = "/root/docs/lanlnk/config/ontology/business-ontology.yaml"
fp = hashlib.sha256(open(ONT_PATH, "rb").read()).hexdigest()[:16]
assert fp == "bf550bc24de66813", "SoT 指纹漂移: " + fp
head = subprocess.run(["git", "-C", "/root/lnkcre", "rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
n_specs = len(os.listdir("/root/lnkcre/openspec/specs"))
print("SoT 指纹复验通过:", fp, "| lnkcre HEAD:", head, "| openspec specs:", n_specs)

gen_terms = json.load(open(os.path.join(CONS_DIR, "term-aliases.generated.json")))
gen_sqls = json.load(open(os.path.join(CONS_DIR, "sql-examples.generated.json")))
SP_DIR = "/root/LnkChatBI/backend/scripts/mall_ops_starter_pack"
sp_terms = json.load(open(os.path.join(SP_DIR, "terminology.json")))
sp_sqls = json.load(open(os.path.join(SP_DIR, "sql_examples.json")))
print("语义包（D3 生成物）:", len(gen_terms), "组术语 /", len(gen_sqls), "条 SQL 示例")
print("starter pack 基线:", len(sp_terms), "组术语 /", len(sp_sqls), "条 SQL 示例")

# D2 教训落地：description 含 XML 特殊字符会破坏 to_xml_string 注入块
bad = [(g["word"], ch) for g in gen_terms for ch in "<>&" if ch in (g["description"] or "")]
bad += [(e["question"][:10], ch) for e in gen_sqls for ch in "<>&" if ch in (e["description"] or "")]
print("XML 特殊字符检查:", ("发现 " + str(bad) + " → 生成时需过滤") if bad else "全部干净（无需过滤）")

## §1 白名单合规复查：11 条 SQL 示例落在哪个对象宇宙

D5 digest 的 P0 发现（`chatbi-governed-account-boundary`）：生产消费面 = `lnk_chatbi_ro` 治理账户恰好持有的
20 个 analysis 对象（16 open + 3 gov 视图 + 1 gov 函数），白名单外新对象默认不可见。
**D5 留下的复查任务：导入前逐条核对 11 条 SQL 引用的对象。**

In [ ]:
WL_SPEC = "/root/lnkcre/openspec/specs/chatbi-governed-account-boundary/spec.md"
wl_text = open(WL_SPEC).read()
WL_TABLES = ["dim_project", "dim_date", "dim_trade", "dim_unit", "dim_store",
             "mv_ops_daily", "mv_store_ops_daily", "mv_collection_summary",
             "mv_customer_receivable_daily", "mv_contract_summary", "mv_leasing_structure",
             "mv_target_tracking", "mv_leads_funnel", "mv_campaign_summary",
             "mv_resource_summary", "mv_property_summary",
             "gov_snap_lease_daily", "gov_snap_lease_monthly", "gov_lease_expiry_summary"]
WL_FUNC = "gov_ar_aging_summary"
wl_ev = {n: (n in wl_text) for n in WL_TABLES + [WL_FUNC]}
assert all(wl_ev.values()), ("白名单对象未在 spec 中找到", wl_ev)
WL = set(WL_TABLES) | {WL_FUNC}
print("生产白名单（%s）: %d 表/视图 + 1 函数，全部在 spec 中验证" % (os.path.basename(WL_SPEC), len(WL_TABLES)))

def sql_tables(sql):
    return sorted(set(m.lower() for m in re.findall(r"(?:FROM|JOIN)\s+([A-Za-z_][A-Za-z0-9_]*)", sql, re.I)))

# mallcre demo 对象宇宙（LnkChatBI 自带 demo 数据源的两份 DDL）
mallcre_text = open("/root/LnkChatBI/mallcre.sql", encoding="utf-8", errors="ignore").read()
demo_text = open("/root/LnkChatBI/postgres_demo_schema.sql").read()
demo_objs = set()
for line in mallcre_text.splitlines():
    if re.match(r"\s*CREATE TABLE", line, re.I):
        name = line.strip().rstrip("(").split()[-1].strip("`\"").lower()
        if name not in ("as", "exists"):
            demo_objs.add(name)
for line in demo_text.splitlines():
    if re.match(r"\s*create (or replace )?(table|view)", line, re.I):
        toks = line.strip().rstrip("(").split()
        name = toks[-1].strip("`\"").lower()
        if name == "as" and len(toks) >= 2:
            name = toks[-2].strip("`\"").lower()
        if name not in ("as", "exists"):
            demo_objs.add(name)
print("mallcre demo 对象宇宙:", len(demo_objs), "个对象")

comp = []
for i, ex in enumerate(gen_sqls, 1):
    tabs = sql_tables(ex["description"])
    comp.append({"no": i, "q": ex["question"][:24], "tabs": tabs,
                 "in_prod": sorted(t for t in tabs if t in WL),
                 "demo_valid": all(t in demo_objs for t in tabs)})
print()
print("No. | 问题(截断)                 | 引用对象                          | 生产白名单 | demo 域合法")
for c in comp:
    print("%2d  | %-26s | %-32s | %-10s | %s" % (c["no"], c["q"], ",".join(c["tabs"]),
          ",".join(c["in_prod"]) if c["in_prod"] else "-", "PASS" if c["demo_valid"] else "FAIL"))
n_prod = sum(1 for c in comp if c["in_prod"])
n_valid = sum(1 for c in comp if c["demo_valid"])
print()
print("结论: 落生产白名单 %d/11 | mallcre demo 域合法 %d/11" % (n_prod, n_valid))
print("解读: 11 条示例按 specific_ds=true 生成于 mallcre demo 数据源（D3 设计），本就不指向生产 analysis 域；")
print("      但 D5 的验收改写揭示的是——demo 与生产是两个对象宇宙，0 交集即『距离生产的第一层』。")

# demo→prod 对象初判映射（W16 白名单绑定生成器的输入；证据=wave1-3 specs 对象语义）
DEMO2PROD = {
    "bi_d_position": ("dim_unit + dim_store", "铺位→租赁单元/店铺维度（wave1 dim_unit、白名单 dim_store）"),
    "bi_b_tenant": ("(无直接对象)", "商户维度未入白名单——只能经 mv_contract_summary 口径，映射缺口①"),
    "bi_d_contract": ("mv_contract_summary / gov_snap_lease_daily", "合同明细→汇总 mv + 治理日快照"),
    "bisubject": ("(无直接对象)", "费用科目维度未入白名单——mv_collection_summary 只留结果，映射缺口②"),
    "bipreddeposit": ("mv_collection_summary(科目子集)", "押金流水→收款汇总，明细粒度丢失，映射缺口③"),
    "bibillrecvinfo": ("mv_customer_receivable_daily / gov_ar_aging_summary", "账单→应收日表+账龄治理函数"),
    "fact_parking_daily": ("(无直接对象)", "停车域生产侧无白名单对象（mv_property_summary 是工单域），映射缺口④"),
    "vw_mall_ops_vacancy_snapshot": ("mv_resource_summary", "空置快照→资源汇总（与 snap_lease_daily 聚合对账）"),
    "vw_mall_ops_contract_expiry": ("gov_lease_expiry_summary", "到期视图→治理汇总"),
}
gap_objs = [k for k, v in DEMO2PROD.items() if "无直接对象" in v[0] or "缺口" in v[1]]
print()
print("demo→prod 初判映射缺口对象:", len(gap_objs), "→", gap_objs)

**图 1：对象宇宙对照**——左列 = 11 条示例实际引用的 demo 对象，右列 = 生产白名单相关对象（节选），
绿线 = 有初判映射，红箭头 = 映射缺口（demo 侧无生产承接对象）。

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 8.5))
left_items = list(DEMO2PROD.keys())
right_relevant = ["dim_unit", "dim_store", "mv_contract_summary", "gov_snap_lease_daily",
                  "mv_collection_summary", "mv_customer_receivable_daily", "gov_ar_aging_summary",
                  "mv_resource_summary", "gov_lease_expiry_summary", "(白名单其余 11 对象)"]
for y, name in enumerate(left_items):
    gap = "无直接对象" in DEMO2PROD[name][0] or "缺口" in DEMO2PROD[name][1]
    ax.text(0.02, 1 - y * 0.105, name, fontsize=11, family=font_name,
            color="#c0392b" if gap else "#1a7a1a", va="center")
for y, name in enumerate(right_relevant):
    ax.text(0.98, 1 - y * 0.105, name, fontsize=11, family=font_name, va="center", ha="right", color="#1f4e9c")
ry = {n: i for i, n in enumerate(right_relevant)}
pairs = [("bi_d_position", "dim_unit", 0), ("bi_d_position", "dim_store", 1),
         ("bi_d_contract", "mv_contract_summary", 0), ("bi_d_contract", "gov_snap_lease_daily", 1),
         ("bibillrecvinfo", "mv_customer_receivable_daily", 0), ("bibillrecvinfo", "gov_ar_aging_summary", 1),
         ("vw_mall_ops_vacancy_snapshot", "mv_resource_summary", 0),
         ("vw_mall_ops_contract_expiry", "gov_lease_expiry_summary", 0),
         ("bipreddeposit", "mv_collection_summary", 0)]
for lname, rname, shift in pairs:
    ly = left_items.index(lname)
    ax.annotate("", xy=(0.64, 1 - ry[rname] * 0.105 - shift * 0.028), xytext=(0.30, 1 - ly * 0.105),
                arrowprops=dict(arrowstyle="-", color="#7aa87a", lw=1.1, alpha=0.85))
for name in ["bi_b_tenant", "bisubject", "fact_parking_daily"]:
    y = 1 - left_items.index(name) * 0.105
    ax.annotate("", xy=(0.58, y), xytext=(0.30, y), arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.5))
    ax.text(0.60, y, "缺口", fontsize=10.5, color="#c0392b", family=font_name, va="center")
ax.text(0.02, 1.05, "demo 数据源（mallcre）对象 × 11 条示例", fontsize=12.5, family=font_name, weight="bold")
ax.text(0.98, 1.05, "生产白名单（lnk_chatbi_ro · analysis 域）", fontsize=12.5, family=font_name, weight="bold", ha="right")
ax.set_title("W15-D6 对象宇宙对照：0/11 交集，4 个映射缺口", fontsize=14, family=font_name, pad=34)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.1); ax.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(NB_DIR, "w15d6_object_universe.png"), dpi=140)
plt.show()
print("图已保存: w15d6_object_universe.png")

## §2 导入物化 + SQLite 镜像导入演练

物化通道选 **starter-pack 同构格式**（`setup_mall_ops_starter_pack.py` 读取的 terminology.json / sql_examples.json，
字段 word/other_words/description + question/description=SQL）——与 LnkChatBI 既有导入机制零适配成本。
服务器无 PG 实例，导入执行面用 **SQLite 镜像**（表结构对齐 `Terminology`/`DataTraining` SQLModel 定义：
术语别名 = 同表父子行 pid 结构），复刻 starter 脚本的 upsert 语义做**幂等收敛证明**。
D3 的 `SET_AT_IMPORT` 在此解析：specific_ds=true + datasource_ids=[1]（CRE BI Demo）。

In [ ]:
DS_NAME, DS_ID = "CRE BI Demo", 1
pack_terms = [{"word": g["word"], "other_words": g["other_words"], "description": g["description"]} for g in gen_terms]
pack_sqls = [{"question": e["question"], "description": e["description"]} for e in gen_sqls]
json.dump(pack_terms, open(os.path.join(PACK_DIR, "terminology.json"), "w"), ensure_ascii=False, indent=1)
json.dump(pack_sqls, open(os.path.join(PACK_DIR, "sql_examples.json"), "w"), ensure_ascii=False, indent=1)
readme = "\n".join([
    "# LnkChatBI import-pack：MI CRE Semantic Model v0.1.1 消费物（W15-D6 物化）",
    "",
    "- 格式：与 backend/scripts/mall_ops_starter_pack/ 同构（terminology.json + sql_examples.json），",
    "  可被 setup_mall_ops_starter_pack.py 同款 upsert 通道消费。",
    "- 作用域：specific_ds=true，datasource_ids=[%d]（%s）——不进 oid 级共享池（W15-D2 决策②）。" % (DS_ID, DS_NAME),
    "- 源头：semantic-model v0.1.1 → W15-D3 生成物（term-aliases.generated.json / sql-examples.generated.json），",
    "  ontology 指纹 %s（SoT 未漂移）。" % fp,
    "- 消费回执：import-receipt.json（S6 消费回执律：manifest + 验证数字 + SoT 指纹）。",
    "- 已知边界：11 条 SQL 全部落在 mallcre demo 对象宇宙，生产白名单合规 = 0/11（设计域使然，见 §1）；",
    "  示例 #2 谓词值域与种子数据字典不一致（POSITION_STATE 枚举 1/2 vs 文本口径）——源头修正回 D3 生成器，",
    "  不在本 pack 静默改写；W16+ 以白名单 20 对象为绑定目标重生成 v0.2 版本。",
])
open(os.path.join(PACK_DIR, "README.md"), "w").write(readme)
print("import-pack 落盘:", sorted(os.listdir(PACK_DIR)))

def make_mirror():
    c = sqlite3.connect(":memory:")
    c.execute("""CREATE TABLE terminology(id INTEGER PRIMARY KEY AUTOINCREMENT, oid INTEGER DEFAULT 1,
     pid INTEGER, word TEXT, description TEXT, specific_ds INTEGER DEFAULT 0,
     datasource_ids TEXT DEFAULT '[]', enabled INTEGER DEFAULT 1)""")
    c.execute("""CREATE TABLE data_training(id INTEGER PRIMARY KEY AUTOINCREMENT, oid INTEGER DEFAULT 1,
     datasource INTEGER, question TEXT, description TEXT, enabled INTEGER DEFAULT 1)""")
    return c

def upsert_terms(conn, terms, ds_id):
    ins_p = ins_c = 0
    for g in terms:
        row = conn.execute("SELECT id FROM terminology WHERE word=? AND pid IS NULL", (g["word"],)).fetchone()
        if row is None:
            cur = conn.execute("INSERT INTO terminology(word,description,specific_ds,datasource_ids) VALUES(?,?,1,?)",
                               (g["word"], g["description"], json.dumps([ds_id])))
            pid = cur.lastrowid; ins_p += 1
        else:
            pid = row[0]
            conn.execute("UPDATE terminology SET description=?, specific_ds=1, datasource_ids=? WHERE id=?",
                         (g["description"], json.dumps([ds_id]), pid))
        have = {r[0] for r in conn.execute("SELECT word FROM terminology WHERE pid=?", (pid,))}
        for a in g["other_words"]:
            if a not in have:
                conn.execute("INSERT INTO terminology(pid,word) VALUES(?,?)", (pid, a)); ins_c += 1
    return ins_p, ins_c

def upsert_examples(conn, sqls, ds_id):
    ins = 0
    for e in sqls:
        row = conn.execute("SELECT id FROM data_training WHERE question=? AND datasource=?",
                           (e["question"], ds_id)).fetchone()
        if row is None:
            conn.execute("INSERT INTO data_training(datasource,question,description) VALUES(?,?,?)",
                         (ds_id, e["question"], e["description"])); ins += 1
    return ins

mir = make_mirror()
a1 = upsert_terms(mir, sp_terms, DS_ID) + (upsert_examples(mir, sp_sqls, DS_ID),)
b1 = upsert_terms(mir, pack_terms, DS_ID) + (upsert_examples(mir, pack_sqls, DS_ID),)
b2 = upsert_terms(mir, pack_terms, DS_ID) + (upsert_examples(mir, pack_sqls, DS_ID),)
print("starter 导入(父组,新别名,新示例):", a1)
print("语义包第一遍(父组,新别名,新示例):", b1)
print("语义包第二遍(父组,新别名,新示例):", b2, "→ 全 0 = 幂等收敛 PASS")
assert b2 == (0, 0, 0), "upsert 不幂等"
n_par = mir.execute("SELECT COUNT(*) FROM terminology WHERE pid IS NULL").fetchone()[0]
n_chi = mir.execute("SELECT COUNT(*) FROM terminology WHERE pid IS NOT NULL").fetchone()[0]
n_ex = mir.execute("SELECT COUNT(*) FROM data_training").fetchone()[0]
print("镜像终态: %d 父组 + %d 子别名 + %d 示例" % (n_par, n_chi, n_ex))
ds_check = mir.execute("SELECT DISTINCT datasource_ids FROM terminology WHERE pid IS NULL AND specific_ds=1").fetchall()
print("作用域检查（specific_ds=1 绑定值）:", ds_check)
assert all(json.loads(d[0]) == [DS_ID] for d in ds_check)

## §3 覆盖率 A/B 复测：21 题电池

检索语义复刻（W15-D1/D2 逐行读码结论）：**术语=单向子串 + 命中任一别名拉全组；示例=双向子串**。
电池四组：Q=11 条示例原问（ERP 对象域）｜V=5 条口语变体｜P=3 条生产白名单域问题｜S=2 条 starter 域守恒题（防回归）。
状态 A（仅 starter）与状态 B（+语义包）各自独立镜像，索引从镜像 SQL 查出——证明导入生效而非内存直读。

In [ ]:
def term_index(conn):
    idx = []
    for pid, w, d, s, ds in conn.execute(
            "SELECT id,word,description,specific_ds,datasource_ids FROM terminology WHERE pid IS NULL"):
        aliases = [r[0] for r in conn.execute("SELECT word FROM terminology WHERE pid=?", (pid,))]
        idx.append({"word": w, "description": d or "", "aliases": aliases})
    return idx

def hit_terms(q, idx):
    return [g for g in idx if (g["word"] in q) or any(a in q for a in g["aliases"])]

def hit_example(q, exs):
    return [(qq, ss) for qq, ss in exs if (q in qq) or (qq in q)]

mirA, mirB = make_mirror(), make_mirror()
upsert_terms(mirA, sp_terms, DS_ID); upsert_examples(mirA, sp_sqls, DS_ID)
upsert_terms(mirB, sp_terms, DS_ID); upsert_examples(mirB, sp_sqls, DS_ID)
pb = upsert_terms(mirB, pack_terms, DS_ID); pe = upsert_examples(mirB, pack_sqls, DS_ID)
idx_A, idx_B = term_index(mirA), term_index(mirB)
exs_A = [(q, s) for q, s in mirA.execute("SELECT question,description FROM data_training")]
exs_B = [(q, s) for q, s in mirB.execute("SELECT question,description FROM data_training")]
print("状态A(基线): %d 组 / %d 示例 | 状态B(+语义包: +%d组 +%d示例): %d 组 / %d 示例" % (
    len(idx_A), len(exs_A), pb[0], pe, len(idx_B), len(exs_B)))

battery = [("Q%d" % (i + 1), "Q", e["question"]) for i, e in enumerate(gen_sqls)]
battery += [("V1", "V", "A101 为什么租不出去？"),
            ("V2", "V", "现在有哪些空着的铺？"),
            ("V3", "V", "星河中心有哪些入驻商户？"),
            ("V4", "V", "合同 CONT_DEMO_001 下面挂了哪些铺？"),
            ("V5", "V", "铺位 L2-02 现在是什么状态？"),
            ("P1", "P", "本月招商线索转化漏斗怎么样？"),
            ("P2", "P", "各项目工单完成率如何？"),
            ("P3", "P", "当前项目租赁结构指标怎么样？"),
            ("S1", "S", "昨日全场销售额、客流、坪效分别是多少？"),
            ("S2", "S", "未来 90 天有哪些合同到期？")]
results = []
for tid, cat, q in battery:
    th, te = hit_terms(q, idx_A), hit_example(q, exs_A)
    bh, be = hit_terms(q, idx_B), hit_example(q, exs_B)
    results.append({"id": tid, "cat": cat, "q": q,
                    "A_term": [g["word"] for g in th], "B_term": [g["word"] for g in bh],
                    "A_ex": len(te) > 0, "B_ex": len(be) > 0})
print()
print("ID  | 类 | 问题                           | 基线:术语/示例        | +语义包:术语/示例")
for r in results:
    print("%-3s | %s | %-30s | %-21s | %s" % (r["id"], r["cat"], r["q"][:30],
          ("/".join(r["A_term"]) or "-") + " / " + ("Y" if r["A_ex"] else "-"),
          ("/".join(r["B_term"]) or "-") + " / " + ("Y" if r["B_ex"] else "-")))
def agg(res, pre):
    t = sum(1 for r in res if r[pre + "_term"]); e = sum(1 for r in res if r[pre + "_ex"])
    return t, e
covA, covB = agg(results, "A"), agg(results, "B")
print()
print("21 题电池汇总: 术语命中 基线 %d → %d | 示例命中 基线 %d → %d" % (covA[0], covB[0], covA[1], covB[1]))
print("守恒检查（S 组防回归）: S1/S2 在 B 态仍命中 →", all(r["B_ex"] for r in results if r["cat"] == "S"))
print("变体发现: V2 全 MISS（口语『空着的铺』未入别名，v0.2 候选）；V1/V3/V4/V5 术语命中但示例 MISS（子串半径外 → LLM 自由组装区）")
assert all(r["B_ex"] for r in results if r["cat"] == "S"), "starter 域回归"

## §3.2 执行面：mallcre 种子装载 + 11 条 SQL 真跑

装载三层：① mallcre_seed_realistic.sql（ERP 镜像 bi_* 表，INSERT 解析）② bipreddeposit（仅 DDL，种子无行 → 诚实 0 行）
③ BI demo schema（PG DDL + 种子：dim/fact + 两视图翻译为 SQLite；`fact_shop_daily_operation` 种子是 CTE 生成——
按其 CASE 规则忠实物化 business_date=2024-12-17 单日快照，`fact_parking_daily`/完整日历 CTE 不复刻 → 0 行）。

In [ ]:
db = sqlite3.connect(":memory:")

def scan_tuples(body):
    rows, cur, depth, q = [], [], 0, False
    for ch in body:
        if q:
            cur.append(ch)
            if ch == "'": q = False
            continue
        if ch == "'":
            cur.append(ch); q = True
        elif ch == "(":
            depth += 1
            if depth == 1: cur = []
            else: cur.append(ch)
        elif ch == ")":
            depth -= 1
            if depth == 0:
                rows.append(cur); cur = []
            else: cur.append(ch)
        else:
            if depth >= 1: cur.append(ch)
    return [r for r in rows if r]

def split_tokens(s):
    toks, buf, q = [], "", False
    for ch in s:
        if q:
            buf += ch
            if ch == "'": q = False
        elif ch == "'":
            buf += ch; q = True
        elif ch == ",":
            toks.append(buf.strip()); buf = ""
        else:
            buf += ch
    if buf.strip(): toks.append(buf.strip())
    return toks

def tok(v):
    if v.startswith("'"): return v[1:-1]
    low = v.lower()
    if low == "null": return None
    if low == "true": return 1
    if low == "false": return 0
    try: return int(v)
    except ValueError: pass
    try: return float(v)
    except ValueError: return v

def load_inserts(text, pattern):
    loaded = {}
    for m in re.finditer(pattern + r"\s*\(([^)]*?)\)\s*values(.*?);", text, re.S | re.I):
        table = m.group(1).strip().strip('"').lower()
        cols = [c.strip().strip('"') for c in m.group(2).split(",")]
        rows = [[tok(t) for t in split_tokens("".join(r))] for r in scan_tuples(m.group(3))]
        if table not in loaded and rows and all(len(r) == len(cols) for r in rows):
            loaded[table] = (cols, rows)
    return loaded

mcre = load_inserts(open("/root/LnkChatBI/mallcre_pg_init/mallcre_seed_realistic.sql").read(),
                    r'INSERT INTO "([A-Za-z0-9_]+)"')
mcre_counts = {}
for t, (cols, rows) in mcre.items():
    db.execute("CREATE TABLE %s (%s)" % (t, ",".join('"%s"' % c for c in cols)))
    db.executemany("INSERT INTO %s VALUES (%s)" % (t, ",".join("?" * len(cols))), rows)
    mcre_counts[t] = len(rows)
print("mallcre ERP 镜像装载:", mcre_counts)

m = re.search(r"CREATE TABLE `bipreddeposit` \((.*?)\n\)", mallcre_text, re.S)
pp_cols = [ln.strip().rstrip(",").split()[0].strip("`") for ln in m.group(1).splitlines()
           if ln.strip().startswith("`")]
db.execute('CREATE TABLE bipreddeposit (%s)' % ",".join('"%s"' % c for c in pp_cols))
print("bipreddeposit DDL 建表:", len(pp_cols), "列（种子无行 → 查询返回 0 行）")

def pg_cols(name):
    mm = re.search(r"create table if not exists %s\s*\((.*?)\n\);" % name, demo_text, re.S | re.I)
    cols = []
    for ln in mm.group(1).splitlines():
        ln = ln.strip().rstrip(",")
        if not ln or ln.split()[0].lower() in ("primary", "unique", "foreign", "constraint", "check", "index"):
            continue
        cols.append(ln.split()[0])
    return cols

seed_text = open("/root/LnkChatBI/postgres_demo_seed.sql").read()
seed_text = re.sub(r"date '([^']*)'", r"'\1'", seed_text)
seed_text = re.sub(r"\bon conflict\b[^;]*?do nothing;", ";", seed_text)
pgseed = load_inserts(seed_text, r"insert into ([a-z_][a-z0-9_]*)")
need_pg = ["dim_project", "dim_floor", "dim_shop", "dim_brand", "fact_leasing_contract", "fact_parking_daily"]
pg_counts = {}
for t in need_pg:
    if t not in pgseed:
        db.execute("CREATE TABLE %s (%s)" % (t, ",".join(pg_cols(t)))); pg_counts[t] = 0
        continue
    cols, rows = pgseed[t]
    ddl = pg_cols(t)
    use = ddl if len(ddl) == len(cols) else cols
    db.execute("CREATE TABLE %s (%s)" % (t, ",".join('"%s"' % c for c in use)))
    db.executemany("INSERT INTO %s VALUES (%s)" % (t, ",".join("?" * len(use))), rows)
    pg_counts[t] = len(rows)
print("BI demo schema 装载:", pg_counts)

db.execute("CREATE TABLE fact_shop_daily_operation (%s)" % ",".join('"%s"' % c for c in pg_cols("fact_shop_daily_operation")))
db.execute("CREATE TABLE dim_date (%s)" % ",".join('"%s"' % c for c in pg_cols("dim_date")))
BD, BDK = "2024-12-17", 20241217
db.execute("INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)",
           (BDK, BD, 2024, 4, 12, "2024-12", 51, 17, 2, "Tuesday", 0, 0, 0))
shops = db.execute("SELECT shop_id, project_id, business_category, sub_business_category FROM dim_shop").fetchall()
cont = {r[0]: r for r in db.execute(
    "SELECT shop_id, lease_start_date, lease_end_date, handover_date, opening_date, closing_date, brand_id FROM fact_leasing_contract")}
ops_rows, vacant_shops = [], []
for sid, pid, cat, sub in shops:
    c = cont.get(sid)
    ls, le, hd, od, cd, bid = (c[1], c[2], c[3], c[4], c[5], c[6]) if c else (None,) * 6
    if ls is None or BD < ls or BD > le: st = "vacant"
    elif cd is not None and BD >= cd: st = "closed"
    elif hd is not None and BD < hd: st = "pre_open"
    elif od is None or BD < od: st = "fitout"
    else: st = "operating"
    if st == "vacant": vacant_shops.append(sid)
    ops_rows.append((BDK * 10000 + sid, BDK, pid, sid, bid, st, 0, 0, 0, 0, 0, 0, 0, 0, 0))
db.executemany("INSERT INTO fact_shop_daily_operation VALUES (%s)" % ",".join("?" * 15), ops_rows)

db.execute("CREATE TABLE vw_mall_ops_demo_context (business_date TEXT, business_date_key INTEGER, yesterday_date TEXT, yesterday_date_key INTEGER, week_start_date TEXT, week_start_date_key INTEGER, week_end_date TEXT, week_end_date_key INTEGER, future_90_days_end_date TEXT)")
db.execute("INSERT INTO vw_mall_ops_demo_context VALUES ('2024-12-17',20241217,'2024-12-16',20241216,'2024-12-16',20241216,'2024-12-22',20241222,'2025-03-17')")
db.execute("""CREATE VIEW vw_mall_ops_vacancy_snapshot AS
SELECT o.date_key, d.calendar_date, o.project_id, p.project_name, f.floor_code, f.floor_name,
 s.shop_id, s.shop_code, s.shop_name, s.gla_area AS vacant_area, s.business_category, s.sub_business_category
FROM fact_shop_daily_operation o
CROSS JOIN vw_mall_ops_demo_context ctx
JOIN dim_date d ON d.date_key = o.date_key
JOIN dim_project p ON p.project_id = o.project_id
JOIN dim_shop s ON s.shop_id = o.shop_id
JOIN dim_floor f ON f.floor_id = s.floor_id
WHERE o.business_status = 'vacant' AND ctx.business_date_key = o.date_key""")
db.execute("""CREATE VIEW vw_mall_ops_contract_expiry AS
SELECT c.contract_id, c.contract_no, c.project_id, p.project_name, s.shop_code, s.shop_name,
 b.brand_name, c.contract_status, c.lease_start_date, c.lease_end_date,
 CAST(julianday(c.lease_end_date) - julianday(ctx.business_date) AS INTEGER) AS days_to_expiry
FROM fact_leasing_contract c
CROSS JOIN vw_mall_ops_demo_context ctx
JOIN dim_project p ON p.project_id = c.project_id
JOIN dim_shop s ON s.shop_id = c.shop_id
JOIN dim_brand b ON b.brand_id = c.brand_id""")
print("ops 单日快照: %d 店（vacant %d: %s）| 视图 ×2 建成" % (len(ops_rows), len(vacant_shops), vacant_shops))

exec_res = []
for i, ex in enumerate(gen_sqls, 1):
    sql = ex["description"]
    try:
        cur = db.execute(sql); n = len(cur.fetchall()); exec_res.append((i, ex["question"][:26], "OK", n))
    except Exception as err:
        exec_res.append((i, ex["question"][:26], "FAIL:" + str(err)[:40], -1))
print()
print("No. | 问题(截断)                     | 执行 | 行数")
for i, q, s, n in exec_res:
    print("%2d  | %-30s | %-6s | %d" % (i, q, s, n))
n_ok = sum(1 for r in exec_res if r[2] == "OK")
zero_rows = [r[0] for r in exec_res if r[2] == "OK" and r[3] == 0]
print("执行面: %d/11 SQL 合法执行；0 行问题 %s" % (n_ok, zero_rows))
print("发现: 示例 #2 谓词 POSITION_STATE='空置' 与种子数据字典不一致（DDL 注释 1:在租/2:空置）——")
print("      D3 验证器只锚表/列存在性，值域枚举未校验 → 验证器 v0.1 边界登记 + change 候选包+1")
assert n_ok == 11

**图 2：覆盖率 A/B 对比**——21 题检索层（术语/示例命中）+ 11 题执行层（示例命中且 SQL 真跑成功）。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
labels = ["术语命中\n(21题)", "示例命中\n(21题)", "端到端\n(11题·示例+执行)"]
base_vals = [covA[0], covA[1], 0]
after_vals = [covB[0], covB[1], sum(1 for r in exec_res if r[2] == "OK" and results[r[0]-1]["B_ex"])]
x = range(3)
axes[0].bar([i - 0.18 for i in x], base_vals, width=0.34, label="基线（仅 starter pack）", color="#9db3c9")
axes[0].bar([i + 0.18 for i in x], after_vals, width=0.34, label="+语义包导入后", color="#2e7d32")
for i, (b, a) in enumerate(zip(base_vals, after_vals)):
    axes[0].text(i - 0.18, b + 0.3, str(b), ha="center", fontsize=11)
    axes[0].text(i + 0.18, a + 0.3, str(a), ha="center", fontsize=11, weight="bold")
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(labels, family=font_name, fontsize=10)
axes[0].set_ylim(0, 24); axes[0].legend(prop={"family": font_name})
axes[0].set_title("覆盖率 A/B：21 题电池（Q11/V5/P3/S2）", family=font_name, fontsize=13)

cats = ["Q(示例原问)", "V(口语变体)", "P(生产域)", "S(starter域)"]
termB = [sum(1 for r in results if r["cat"] == c[0] and r["B_term"]) for c in cats]
termA = [sum(1 for r in results if r["cat"] == c[0] and r["A_term"]) for c in cats]
tot = [sum(1 for r in results if r["cat"] == c[0]) for c in cats]
y = range(4)
axes[1].barh([i + 0.18 for i in y], termB, height=0.34, color="#2e7d32", label="+语义包")
axes[1].barh([i - 0.18 for i in y], termA, height=0.34, color="#9db3c9", label="基线")
for i, (a, b, t) in enumerate(zip(termA, termB, tot)):
    axes[1].text(b + 0.15, i + 0.18, "%d/%d" % (b, t), va="center", fontsize=11, family=font_name)
    axes[1].text(a + 0.15, i - 0.18, "%d/%d" % (a, t), va="center", fontsize=10, color="#555")
axes[1].set_yticks(list(y)); axes[1].set_yticklabels(cats, family=font_name)
axes[1].set_xlim(0, 13); axes[1].invert_yaxis(); axes[1].legend(prop={"family": font_name})
axes[1].set_title("分域术语命中：V 组 4/5（V2 口语缺口）、P 组 0/3（生产域空白）", family=font_name, fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(NB_DIR, "w15d6_coverage_ab.png"), dpi=140)
plt.show()
print("图已保存: w15d6_coverage_ab.png")

## §4 Demo②：A101 依据链问答（L1-L3 机器复核）

链路：问题 → 术语检索（A101 别名命中『铺位』组 → 全组归并）→ description 注入（FILTER_TERMS 块，
含编码映射规则 + 双口径警示）→ SQL 组装（示例 Q1 为校准 + L2 合同链扩展）→ 种子真跑 → 答案组装 → 分级判定。
判定梯 = W15-D1 定稿的 LnkChatBI 语境验收口径（L0 基线护栏由 D7 裁决，此处 L1-L3）。

In [ ]:
Q_A101 = "A101 铺位为什么不能出租？"
trace = []
th = hit_terms(Q_A101, idx_B)
grp = next(g for g in th if "A101" in g["aliases"])
trace.append(("1.术语检索", "A101 ∈ 『%s』组别名 → 全组归并（%d 别名一起进 prompt）" % (grp["word"], len(grp["aliases"]))))
trace.append(("2.注入块", "FILTER_TERMS: 『%s』description = %s" % (grp["word"], grp["description"][:64] + "…")))
assert "LOC_DEMO_L1xx" in grp["description"], "编码映射规则未随 description 注入"
sql_a101 = ("SELECT p.STORE_NAME, p.BUILDING_NAME, p.FLOOR_NAME, p.POSITION_CODE, p.POSITION_NAME, "
            "p.POSITION_TYPE, p.POSITION_STATE, p.RENT_AREA, p.CONT_NO, p.END_DATE AS POS_END_DATE, "
            "c.TENANT_NAME, c.SIGNBOARD, c.CONT_STATE, c.BEGIN_DATE AS CONT_BEGIN, c.END_DATE AS CONT_END "
            "FROM bi_d_position p LEFT JOIN bi_d_contract c ON p.CONT_NO = c.CONT_NO "
            "WHERE p.POSITION_CODE = 'LOC_DEMO_L101'")
cur = db.execute(sql_a101)
acols = [d[0] for d in cur.description]
arow = cur.fetchone()
assert arow is not None, "A101 主行未命中（别名映射链断裂）"
A = dict(zip(acols, arow))
contrast = db.execute("SELECT POSITION_CODE, POSITION_NAME, POSITION_STATE, CONT_NO, END_DATE FROM bi_d_position "
                      "WHERE POSITION_CODE = 'LOC_DEMO_L202'").fetchone()
answer = ("A101（编码 %s，%s，%s/%s/%s）当前不能出租：已有有效租约 %s（租户 %s，租期 %s ~ %s，合同状态=%s，铺位状态=在租[POSITION_STATE=1]）。"
          "对照证据：同项目 %s（%s）为空置状态[POSITION_STATE=2]、CONT_NO='-'、END_DATE=%s，属可招商铺位。"
          "口径提示（术语 description 注入）：铺位空置判定存在双口径——bi_d_position.POSITION_STATE 与 "
          "vw_mall_ops_vacancy_snapshot.business_status，跨口径统计需先对齐。") % (
    A["POSITION_CODE"], A["POSITION_NAME"], A["STORE_NAME"], A["BUILDING_NAME"], A["FLOOR_NAME"],
    A["CONT_NO"], A["TENANT_NAME"], str(A["CONT_BEGIN"])[:10], str(A["CONT_END"])[:10], A["CONT_STATE"],
    contrast[0], contrast[1], str(contrast[4])[:10])
trace.append(("3.SQL组装", "示例Q1校准 + L2扩展（LEFT JOIN bi_d_contract ON CONT_NO）"))
trace.append(("4.种子真跑", "主行 %s；对照行 %s" % (A["POSITION_CODE"], contrast[0])))
trace.append(("5.答案组装", answer[:80] + "…"))
for step, msg in trace:
    print("◆ %s\n  %s\n" % (step, msg))

l1 = ("POSITION_CODE = 'LOC_DEMO_L101'" in sql_a101 and "bi_d_position" in sql_a101
      and all(k in acols for k in ["STORE_NAME", "BUILDING_NAME", "FLOOR_NAME", "POSITION_CODE", "POSITION_NAME"])
      and all(A[k] for k in ["STORE_NAME", "BUILDING_NAME", "FLOOR_NAME", "POSITION_CODE"]))
l2 = ("bi_d_contract" in sql_a101 and "ON p.CONT_NO = c.CONT_NO" in sql_a101
      and A["CONT_NO"] == "CONT_DEMO_001" and A["TENANT_NAME"] == "云巷咖啡"
      and "POSITION_STATE" in sql_a101 and "END_DATE" in sql_a101
      and str(contrast[2]) == "2" and contrast[3] == "-")
l3 = ("双口径" in answer and "POSITION_STATE" in answer and "business_status" in answer
      and "双口径" in grp["description"])
print("L1 身份路径（谓词+四级路径列+真实行值）:", "PASS" if l1 else "FAIL")
print("L2 业务链（CONT_NO→合同/租户，状态/日期引用，对照行空置证据）:", "PASS" if l2 else "FAIL")
print("L3 规则引用（description 注入 + 答案引用双口径条件）:", "PASS" if l3 else "FAIL")
assert l1 and l2 and l3, "L1-L3 必达线未过"
print()
print("L4 Policy 判断: TODO —— D4 约束声明 v0.1-draft 未评审（W18 ⑧），不得消费")
print("L5 动作建议  : TODO —— 超出 Text-to-SQL 形态（capability 执行姿态属数字员工 2.0）")
print("L1' 权限护栏 : TODO —— 需真实行权限栈；D1 已证权限下推为概率性执行（change 候选包）")

**图 3：A101 依据链判定梯**——L1/L2/L3 实证 PASS（绿），L4/L5/L1' 显式 TODO（灰），
每级标注证据源。通过线 = L1+L2+L3（对齐 PT-W4『L1-L3 是地板』）。

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.5))
rungs = [
    ("L1 语义理解·身份路径", True, "SQL 命中 bi_d_position + POSITION_CODE 谓词（A101→LOC_DEMO_L101 别名映射）+ 四级路径列"),
    ("L2 业务链·关系+状态", True, "CONT_NO→bi_d_contract→租户；POSITION_STATE/END_DATE 引用；对照行 L2-02 空置证据"),
    ("L3 规则·口径注入", True, "『铺位』description 双口径警示注入并被答案引用（Rule 构件 L3 接口实证）"),
    ("L4 Policy 判断", None, "TODO：AI 执行约束声明 v0.1-draft 未评审（W15-D4 → W18 ⑧）"),
    ("L5 动作建议", None, "TODO：capability 执行姿态超出 Text-to-SQL 形态"),
    ("L1' 权限护栏", None, "TODO：行权限下推概率性执行 gap（D1 实证，主仓 change 候选）"),
]
for i, (name, st, ev) in enumerate(rungs):
    y = len(rungs) - 1 - i
    color = "#2e7d32" if st else "#9e9e9e"
    ax.add_patch(plt.Rectangle((0.01, y - 0.36), 0.30, 0.72, color=color, alpha=0.88))
    ax.text(0.16, y, name, ha="center", va="center", color="white", fontsize=11.5,
            family=font_name, weight="bold")
    ax.text(0.345, y, ("PASS  " if st else "TODO  ") + ev, va="center", fontsize=10.5, family=font_name)
ax.axhline(2.5, color="#c0392b", lw=1.6, ls="--")
ax.text(0.01, 2.62, "↑ 通过线（L1-L3 地板）｜↓ 加分/超额/护栏（不达标不算 Demo 失败，进 Gap 列表）",
        fontsize=10, color="#c0392b", family=font_name)
ax.set_xlim(0, 1); ax.set_ylim(-0.7, len(rungs) - 0.2); ax.axis("off")
ax.set_title("Demo② A101『为什么不能出租』判定梯：L1-L3 全过，依据链落 chat record 可复核证据",
             fontsize=13.5, family=font_name, pad=14)
plt.tight_layout()
fig.savefig(os.path.join(NB_DIR, "w15d6_a101_ladder.png"), dpi=140)
plt.show()
print("图已保存: w15d6_a101_ladder.png")

## §5 生产差距五层分解（Today's Question）+ S6 消费回执

答案框架：不看功能像不像，看**每一层的失败模式在 demo 里是否已被消掉**。
五层全部有今日实证数字，随后把导入事实 + 验证数字 + SoT 指纹落成消费回执（S6 律）。

In [ ]:
layers = [
    ("①对象宇宙层", "11/11 SQL 落 mallcre demo 域（bi_*），生产白名单 0/11 交集，4 个映射缺口对象",
     "以白名单 20 对象为绑定目标重生成（W16 ③ 登记 Identity 构件后）"),
    ("②数据与编码层", "A101→LOC_DEMO_L1xx 是 demo 种子映射规则；fact_parking/日历 CTE 未复刻（0 行）；示例 #2 值域口径不一致",
     "铺位编码规则注册表 + 值域枚举校验 + 全量种子与真实数据量校验"),
    ("③权限与护栏层", "L1' 未测：需 lnk_chatbi_ro + 行权限账号 + LLM 真栈；D1 已证权限下推概率性执行",
     "sqlglot AST 表提取 + 谓词存在性断言（change 候选包已立）"),
    ("④组装确定性层", "V1/V3/V4/V5 术语命中但示例 MISS（子串半径外=LLM 自由组装区）；V2 全 MISS",
     "示例库扩容 + V2 类口语别名补录（v0.2）+ AST 加固后断言"),
    ("⑤治理与同步层", "导入已带 manifest/指纹/回执（S6 今日闭环），但 SoT 治理头 G-01 未过 change；主仓又在新演进",
     "W16 ① 治理头 change + S1 周一正式 digest（核查第二个 ontology.yaml）"),
]
print("层            | Demo 现状（今日实证）                                      | 生产要求")
for name, now, need in layers:
    print("%-12s | %-58s | %s" % (name, now, need))

receipt = {
    "imported_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "executor": "第15周-Day6 ipynb（SQLite 镜像导入演练；服务器无 PG，LnkChatBI 真库导入为 W16 候选项）",
    "datasource": {"name": DS_NAME, "id": DS_ID, "binding": "specific_ds=true（解析 SET_AT_IMPORT）"},
    "pack": {"terms": len(pack_terms), "aliases": sum(len(g["other_words"]) for g in pack_terms),
             "sql_examples": len(pack_sqls),
             "fingerprints": {f: hashlib.sha256(open(os.path.join(PACK_DIR, f), "rb").read()).hexdigest()[:16]
                              for f in ["terminology.json", "sql_examples.json"]}},
    "upsert": {"first_run": list(b1), "second_run": list(b2), "convergence": "PASS（全 0）"},
    "coverage_ab": {"battery": len(results),
                    "term_base": covA[0], "term_after": covB[0],
                    "example_base": covA[1], "example_after": covB[1],
                    "e2e_exec_ok": n_ok, "e2e_with_example": after_vals[2]},
    "a101_chain": {"L1": "PASS", "L2": "PASS", "L3": "PASS", "L4": "TODO", "L5": "TODO", "L1_prime": "TODO"},
    "whitelist_compliance": {"in_prod_whitelist": "%d/11（demo 数据源设计域）" % n_prod,
                             "mallcre_valid": "%d/11" % n_valid,
                             "mapping_gap_objects": gap_objs},
    "known_gaps_found_today": ["示例#2 值域枚举未校验（POSITION_STATE '空置' vs 种子 1/2）",
                                "V2 口语变体（空着的铺）全 MISS → 别名补录候选"],
    "source": {"ontology_sha256_16": fp, "semantic_model": "v0.1.1",
               "lnkcre_head": head, "starter_pack": SP_DIR},
}
json.dump(receipt, open(os.path.join(PACK_DIR, "import-receipt.json"), "w"), ensure_ascii=False, indent=1)
print()
print("S6 消费回执落盘:", os.path.join(PACK_DIR, "import-receipt.json"))
print(json.dumps(receipt["coverage_ab"], ensure_ascii=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 6))
names = [l[0] for l in layers]
evid = ["0/11 白名单交集\n4 缺口对象", "映射规则=demo 种子域\n值域口径差 ×1", "L1' 未测\n概率性 gap 在案",
        "4 变体入自由组装区\nV2 全 MISS", "S6 闭环\nG-01 未过 change"]
heights = [5, 4, 4, 3, 2]
xs = range(len(layers))
ax.bar(xs, heights, color=["#c0392b", "#d35400", "#d35400", "#e67e22", "#f0c419"], width=0.55)
for i, ev in enumerate(evid):
    ax.text(i, heights[i] + 0.15, ev, ha="center", fontsize=10.5, family=font_name)
ax.set_xticks(list(xs))
ax.set_xticklabels([n.replace("层", "\n层") for n in names], family=font_name, fontsize=11)
ax.set_ylabel("距生产的差距（今日证据强度·示意）", family=font_name, fontsize=11)
ax.set_ylim(0, 6.6)
ax.set_title("Today's Question 答案图：Demo → 生产的五层差距（每层都有今日实证数字）",
             family=font_name, fontsize=13.5, pad=12)
for i, (name, now, need) in enumerate(layers):
    ax.text(i, 0.35, "生产要求：" + need[:13] + "…", ha="center", fontsize=8.5,
            family=font_name, color="#333")
plt.tight_layout()
fig.savefig(os.path.join(NB_DIR, "w15d6_production_gap.png"), dpi=140)
plt.show()

for f in ["terminology.json", "sql_examples.json", "README.md", "import-receipt.json"]:
    assert os.path.exists(os.path.join(PACK_DIR, f)), f
for f in ["w15d6_object_universe.png", "w15d6_coverage_ab.png", "w15d6_a101_ladder.png", "w15d6_production_gap.png"]:
    assert os.path.exists(os.path.join(NB_DIR, f)), f
print("终局断言 PASS：import-pack 四件 + 图×4 落盘完成")